In [4]:
# 9 октября 20:19 - 3 балла
using Random
using LinearAlgebra
using Polynomials
using Plots

function generate_symmetric_matrix(l::Float64, r::Float64, n::Int)
    A = rand(n, n) .* (r - l) .+ l
    return (A + A') / 2
end

function euclidean_norm(vec::Vector)
    return sqrt(sum(vec.^2))
end

function danilevsky_algo(A::Matrix{Float64})
    n = size(A, 1)
    D  = copy(A)
    Bs = Matrix{Float64}(I, n, n)

    for i in n:-1:2
        d = D[i, i-1]
        r = copy(D[i, :])

        newrow = d .* D[i-1, :]
        for j in 1:n
            if j != i-1
                newrow .+= r[j] .* D[j, :]
            end
        end
        D[i-1, :] .= newrow

        col = copy(D[:, i-1])
        D[:, i-1] .= col ./ d
        for k in 1:n
            if k != i-1
                D[:, k] .-= (r[k] / d) .* col
            end
        end

        colB = copy(Bs[:, i-1])
        Bs[:, i-1] .= colB ./ d
        for k in 1:n
            if k != i-1
                Bs[:, k] .-= (r[k] / d) .* colB
            end
        end
    end
    return Bs, D
end

function characteristic_coeffs_from_frobenius(F::Matrix{Float64})
    v = -copy(F[1, :])
    return vcat(1.0, v)
end

companion_eigvec(λ, n) = [λ^(n-1-k) for k in 0:n-1]

function find_eigen_vectors_danilevsky(vals::Vector{Float64}, Bs::Matrix{Float64})
    n = size(Bs, 1)
    m = length(vals)
    X = Matrix{Float64}(undef, n, m)
    for j in 1:m
        y = companion_eigvec(vals[j], n)
        X[:, j] = Bs * y
        X[:, j] ./= norm(X[:, j])
    end
    return X
end

function find_equatation_coeffs(P::Matrix)
    equatationCoeffs = copy(P[1,1:end]).*(-1)
    equatationCoeffs = [1.0; equatationCoeffs]
    return equatationCoeffs
end

function equatation_polynomy(a::Vector, x)
    val = 0
    n = length(a)
    for i in n-1:-1:0
        val += a[n - i] * (x^i)
    end
    return val
end

# Метод бисекции
function find_eigen_values(eqCoeffs::Vector, intervals::Vector, step)
    values = []
    eps = 1e-7
    for interval in intervals
        left = interval[1]
        right = interval[2]
        l = Int(floor((right - left) / step))
    
        for i in 0:l
            x_left = left + i * step
            x_right = x_left + step
            y_left = equatation_polynomy(eqCoeffs, x_left)
            y_right = equatation_polynomy(eqCoeffs, x_right)
            alpha = y_left * y_right

            if alpha < 0
                while x_right - x_left >= eps
                    x_middle = (x_right + x_left) / 2
                    y_middle = equatation_polynomy(eqCoeffs, x_middle)
                    beta = y_left * y_middle
                    if beta < 0
                        x_right = x_middle
                    else
                        x_left = x_middle
                    end
                end
                root = (x_right + x_left) / 2
                if all(r -> abs(r - root) > eps, values)
                    push!(values, root)
                end
            elseif abs(y_left) < eps
                if all(r -> abs(r - x_left) > eps, values)
                    push!(values, x_left)
                end
            elseif abs(y_right) < eps
                if all(r -> abs(r - x_right) > eps, values)
                    push!(values, x_right)
                end
            end
        end
    end
    
    unique_values = unique(round.(values, digits=6))
    return sort(unique_values)[1:min(length(unique_values), length(eqCoeffs)-1)]
end

function check_orthogonal_cols(X; tol=1e-6)
    for i in 1:size(X,2)-1, j in i+1:size(X,2)
        s = dot(X[:, i], X[:, j])
        if abs(s) > tol
            println("Векторы i=$i и j=$j НЕ ортогональны")
            return false
        end
    end
    println("Собственные векторы ортогональны")
    return true
end

function gershgorin_intervals(A::Matrix{Float64})
    n = size(A, 1)
    intervals = Vector{Tuple{Float64,Float64}}()
    for i in 1:n
        r = sum(abs.(A[i, :])) - abs(A[i, i])
        c = A[i, i]
        push!(intervals, (c - r, c + r))
    end
    return intervals
end


function check_vieta(A::Matrix, vals::Vector)
    sum_eigs = sum(vals)
    sp = sum(diag(A))
    if abs(sum_eigs - sp) > 1e-6
        println("\nПроверка Виета: не выполняется")
    else
        println("\nПроверка Виета: выполняется")
    end
end

function check_gershorin(vals::Vector, intervals::Vector)
    minL = minimum(first.(intervals))
    maxR = maximum(last.(intervals))

    bad_vals = [v for v in vals if v < minL || v > maxR]
    if !isempty(bad_vals)
        println("Проверка Гершгорина: ошибка")
        for v in bad_vals
            println("  λ = ", v)
        end
    else
        println("Проверка Гершгорина: выполняется")
    end
end

function plot_gershgorin_characteristic(A::Matrix{Float64};
    num_points::Int=2000, margin::Float64=0.1, digits::Int=4)

    println("Матрица A =")
    display(A)

    # Данилевский → коэффициенты
    Bs, F = danilevsky_algo(A)
    coeffs = characteristic_coeffs_from_frobenius(F)

    println("\nФорма Фробениуса:"); display(F)
    println("\nКоэффициенты характеристического многочлена:")
    println(round.(coeffs; digits=digits))

    # Собственные значения/векторы
    λs = find_eigen_values(Symmetric(A)) |> collect
    println("\nСобственные значения λ:"); println(round.(λs; digits=digits))
    vecs = find_eigen_vectors(real(λs), Bs)
    println("\nСобственные векторы (нормированные):")
    for j in 1:size(vecs, 2)
        println("v_$j = ", round.(vecs[:, j]; digits=digits))
    end
    println(); check_orthogonal_cols(vecs)

    ints = gershgorin_intervals(A)
    xmin = minimum(first.(ints)); xmax = maximum(last.(ints))
    total_interval = (xmin, xmax)

    println("\nИнтервалы Гершгорина:")
    for (k, (l, r)) in enumerate(ints)
        println("I$k = [", round(l; digits=digits), ", ", round(r; digits=digits), "]")
    end
    println("Общий интервал: [", round(xmin; digits=digits), ", ", round(xmax; digits=digits), "]")

    L = xmax - xmin; xmin -= margin*L; xmax += margin*L
    xs = range(xmin, xmax, length=num_points)
    ys = [poly_eval(coeffs, x) for x in xs]

    plt = plot(xs, ys, xlabel="λ", ylabel="p(λ)=det(λI - A)",
               label="p(λ)", title="Характеристический многочлен и интервалы Гершгорина")
    hline!([0.0], label="ось x", linestyle=:dot)

    for (k, (l, r)) in enumerate(ints)
        plot!([l, r], [0.0, 0.0], linewidth=5, label="I$k", alpha=0.9)
    end
    
    display(plt)
    return plt, λs, vecs, ints, total_interval
end

plot_gershgorin_characteristic (generic function with 1 method)

In [3]:
function krylov_algo(matrix::Matrix{Float64})
    p = []
    n = size(matrix, 1)
    D = copy(matrix)
    y = ones(n+1, n)
    A = zeros(n, n)

    for i in 2:n+1
        y[i, 1:end] = D * y[i - 1, 1:end]  # y⁽ᵏ⁾ = A·y⁽ᵏ⁻¹⁾
    end

    for i in n:-1:1
        A[n-i+1, 1:end] = y[i, 1:end]
    end

    A = A'
    f = y[n+1, 1:end]
    
    res_p = A \ f

    p = [1.0]
    for pp in res_p
        push!(p, -pp)
    end
    
    return y, p
end

function find_eigen_vectors_krylov(y, vals, p)
    n = length(p) - 1
    xs = zeros(n,n)
    for i in 1:n
        x = y[n,1:end] # нач вектор Aⁿ⁻¹y
        q_i = [1.0] # q₁ = 1
        for j in 2:n
            push!(q_i,vals[i] * q_i[j - 1] + p[j]) #  # qⱼ = λᵢ × qⱼ₋₁ + pⱼ
            x = x + q_i[j]* y[n - j+1,1:end] # # x = x + qⱼ × yₙ₋ⱼ₊₁
        end
        xs[i,1:end]= x./ euclidean_norm(x)
    end
    return xs
end

n = 4
A = [4.0 1.0 0.5 0.2;
     1.0 3.0 0.8 0.3;
     0.5 0.8 2.0 0.4;
     0.2 0.3 0.4 1.0]

println("Матрица A:")
display(A)

# интервалы Гершгорина
intervals = gershgorin_intervals(A)
println("\nИнтервалы Гершгорина:")
println(intervals)

println("\n" * "="^60)
println("МЕТОД ДАНИЛЕВСКОГО")
println("="^60)

# Получаем коэффициенты характеристического многочлена методом Данилевского
Bs_danil, F_danil = danilevsky_algo(A)
println("Форма Фробениуса:")
display(round.(F_danil, digits=6))

coeffs_danil = characteristic_coeffs_from_frobenius(F_danil)
println("\nКоэффициенты характеристического многочлена:")
println(round.(coeffs_danil, digits=6))

λs_danil = find_eigen_values(coeffs_danil, intervals, 0.01)
println("\nСобственные значения (метод Данилевского + бисекция):")
println(round.(λs_danil, digits=6))

eigenvectors_danil = find_eigen_vectors_danilevsky(real(λs_danil), Bs_danil)
println("\nСобственные векторы (метод Данилевского):")
for i in 1:n
    println("v_$i = ", round.(eigenvectors_danil[i,:], digits=6))
end

println("\n" * "="^60)
println("МЕТОД КРЫЛОВА")
println("="^60)

y_krylov, p_krylov = krylov_algo(A)
println("Коэффициенты характеристического многочлена (Крылов):")
println(round.(p_krylov, digits=17))

λs_krylov = find_eigen_values(p_krylov, intervals, 0.01)
println("\nСобственные значения (метод Крылова + бисекция):")
println(round.(λs_krylov, digits=17))

# Используем правильную функцию для векторов Крылова
eigenvectors_krylov = find_eigen_vectors_krylov(y_krylov, real(λs_krylov), p_krylov)
println("\nСобственные векторы (метод Крылова):")
for i in 1:n
    println("v_$i = ", round.(eigenvectors_krylov[i,:], digits=17))
end

println("\n" * "="^60)
println("СРАВНЕНИЕ МЕТОДОВ")
println("="^60)

println("\nСравнение собственных значений:")
println("Данилевский: ", round.(λs_danil, digits=17)) 
println("Крылов:     ", round.(λs_krylov, digits=17))

println("\nСравнение собственных векторов:")

function sort_eigenpairs(λs, vecs)
    idx = sortperm(real(λs))
    return λs[idx], vecs[idx,:]'
end

λs_danil_sorted, vecs_danil_sorted = sort_eigenpairs(λs_danil, eigenvectors_danil)
λs_krylov_sorted, vecs_krylov_sorted = sort_eigenpairs(λs_krylov, eigenvectors_krylov)
check_orthogonal_cols(vecs_krylov_sorted)
for i in 1:n
    println("\nСобственный вектор $i:")
    println("λ Данилевский: ", round(λs_danil_sorted[i], digits=17))
    println("λ Крылов:     ", round(λs_krylov_sorted[i], digits=17))
    println("Данилевский: ", round.(vecs_danil_sorted[:, i], digits=17))
    println("Крылов:     ", round.(vecs_krylov_sorted[:, i], digits=17))
    
    similarity = abs(dot(vecs_danil_sorted[:, i], vecs_krylov_sorted[:, i]))
    println("Сходство: ", round(similarity, digits=17))
end

Матрица A:


4×4 Matrix{Float64}:
 4.0  1.0  0.5  0.2
 1.0  3.0  0.8  0.3
 0.5  0.8  2.0  0.4
 0.2  0.3  0.4  1.0


Интервалы Гершгорина:
[(2.3, 5.7), (0.9000000000000004, 5.1), (0.30000000000000027, 3.6999999999999997), (0.10000000000000009, 1.9)]

МЕТОД ДАНИЛЕВСКОГО
Форма Фробениуса:


4×4 Matrix{Float64}:
 10.0  -32.82  42.132  -17.7701
  1.0    0.0    0.0      0.0
  0.0    1.0    0.0      0.0
  0.0    0.0    1.0      0.0


Коэффициенты характеристического многочлена:
[1.0, -10.0, 32.82, -42.132, 17.7701]

Собственные значения (метод Данилевского + бисекция):
[0.859381, 1.611422, 2.606932, 4.922265]

Собственные векторы (метод Данилевского):
v_1 = [-0.005941, 0.038841, -0.629373, 0.77611]
v_2 = [-0.009211, -0.543559, 0.638429, 0.544854]
v_3 = [-0.322825, 0.795103, 0.418634, 0.297223]
v_4 = [0.946395, 0.266169, 0.145064, 0.11156]

МЕТОД КРЫЛОВА
Коэффициенты характеристического многочлена (Крылов):
[1.0, -9.999999999999796, 32.81999999999838, -42.1319999999966, 17.770099999998124]

Собственные значения (метод Крылова + бисекция):
[0.859381, 1.611422, 2.606932, 4.922265]

Собственные векторы (метод Крылова):
v_1 = [0.00593990930159237, 0.00921228942609952, 0.32282317469143534, -0.9463958205123064]
v_2 = [0.03884052749857064, -0.5435584736570381, 0.7951036967200642, 0.26616857553316114]
v_3 = [0.629372806753842, -0.6384290236348027, -0.4186343079809416, -0.14506401373470154]
v_4 = [0.7761095490176946, 0.5448